# Free embeddings and custom embeddings

In [2]:
import lancedb
import pandas as pd 

df = pd.read_json("jokes.json")
db = lancedb.connect("knowledge_base")
df.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


## Sentence transformer

In [7]:
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

embedding_model = (
    get_registry()
    .get("sentence-transformers")
    .create(name="intfloat/multilingual-e5-small", device="cpu")
)


class JokeModel(LanceModel):
    joke: str = embedding_model.SourceField()
    vector: Vector(embedding_model.ndims()) = embedding_model.VectorField()


print(f"{embedding_model.ndims()=}")

table = db.create_table(
    "jokes_sentence_transformer", schema=JokeModel, mode="overwrite"
)
table.add(df)

embedding_model.ndims()=384


AddResult(version=3)

In [8]:
table.search("kemi skämt").limit(5).to_pandas()

,joke,vector,_distance
0,Chemists have all the solutions… mostly in bea...,"[0.025907, -0.022183, -0.06204, -0.036586, 0.0...",0.357881
1,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.361319
2,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.372850
3,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.374649
4,"My pipeline failed again, so I’m logging out—l...","[0.014553, -0.021881, 0.002843, -0.057108, 0.0...",0.391870


In [9]:
table.search("math jokes on you").limit(5).to_pandas()

,joke,vector,_distance
0,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0....",0.277347
1,"Dear algebra, please stop asking us to find yo...","[0.05175, 0.00825, -0.020384, -0.038882, 0.084...",0.297305
2,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.328977
3,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.340595
4,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.347134


## Custom embedder

In [18]:
from custom_embeddings.nvidia_nemotron import NvidiaNemotronEmbedding

embedding_model = get_registry().get("nvidia-nemotron").create()

class JokeModel(LanceModel):
    joke: str = embedding_model.SourceField()
    vector: Vector(embedding_model.ndims()) = embedding_model.VectorField()


table_jokes = db.create_table("jokes_nvidia", schema=JokeModel, mode="overwrite")
table_jokes.add(df)

[2026-11-09T11:57:01Z WARN  lance::dataset::write::insert] No existing dataset at /Users/aigineer/Documents/github/youtube_demos/lancedb_custom_embeds/knowledge_base/jokes_nvidia.lance, it will be created


AddResult(version=2)

In [19]:
table_jokes.search("kemiskämt").limit(5).to_pandas()

,joke,vector,_distance
0,I told a chemistry joke… there was no reaction.,"[0.005886, -0.082947, 0.019272, 0.000912, 0.02...",1.583297
1,Why did the chemist ground his kids? Because t...,"[0.000951, -0.075073, 0.017822, 0.00642, 0.010...",1.583896
2,Chemists have all the solutions… mostly in bea...,"[0.002228, -0.099121, 0.020493, 0.021805, 0.01...",1.623199
3,Oxygen and Magnesium got together—OMg!,"[-0.011795, -0.058441, 0.016251, -0.004665, 0....",1.702945
4,Never trust an atom—they make up everything.,"[0.010674, -0.074829, 0.017456, 0.009392, 0.01...",1.744579


In [20]:
table_jokes.search("math jokes").limit(5).to_pandas()

,joke,vector,_distance
0,My math teacher said I’m average… how mean!,"[0.010635, -0.061951, 0.010033, -0.00795, 0.02...",1.199771
1,I told a chemistry joke… there was no reaction.,"[0.005886, -0.082947, 0.019272, 0.000912, 0.02...",1.203905
2,Why’s 6 afraid of 7? Because 7 8 9.,"[-0.005344, -0.05307, 0.00255, -0.018265, 0.02...",1.264562
3,"Gold walks into a bar. The bartender says, “Au...","[-0.004944, -0.056854, 0.010658, -0.011421, 0....",1.292160
4,Why was the equal sign so humble? It knew it w...,"[-0.008629, -0.060516, 0.003263, 0.012245, 0.0...",1.316823


In [22]:
table_jokes.search("math jokes on you").limit(5).to_pandas()

,joke,vector,_distance
0,My math teacher said I’m average… how mean!,"[0.010635, -0.061951, 0.010033, -0.00795, 0.02...",1.205128
1,I told a chemistry joke… there was no reaction.,"[0.005886, -0.082947, 0.019272, 0.000912, 0.02...",1.253349
2,"Dear algebra, please stop asking us to find yo...","[0.011032, -0.091125, 0.021072, 0.007797, 0.02...",1.270159
3,Why’s 6 afraid of 7? Because 7 8 9.,"[-0.005344, -0.05307, 0.00255, -0.018265, 0.02...",1.293273
4,"Gold walks into a bar. The bartender says, “Au...","[-0.004944, -0.056854, 0.010658, -0.011421, 0....",1.337612
